In [16]:
# ============================================================
# STUDENT GRADE REPORT - UPLOAD, SELECT & DOWNLOAD (GREEN & GOLD GLASS UI)
# ------------------------------------------------------------
# Same engine as before (multi-sheet workbooks, title rows above headers,
# Records-sheet preference, GWA / Dean's Lister logic, HTML report, CSV exports)
# with a Notre Dame of Marbel University inspired green & gold glass interface:
#
#   ┌───────────────────────────────────────────────────────────┐
#   │ Header                                                    │
#   ├──────────────┬────────────────────────────────────────────┤
#   │ SIDEBAR      │ [ Student report | Class summary | Data ]  │
#   │  • Upload    │                                            │
#   │  • Student   │  KPI tiles (GWA, units, subjects, ...)     │
#   │    picker    │  Grade table                               │
#   │  • Profile   │  Download button                           │
#   │  • Details   │                                            │
#   └──────────────┴────────────────────────────────────────────┘
#
# Runs in Google Colab or plain Jupyter (ipywidgets).
# After upload, current_file["df"] holds the detected grade-records table.
# ============================================================

import io
import csv
import re
import warnings
import html as html_lib
from datetime import datetime
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML

warnings.filterwarnings("ignore", message="on_submit is deprecated.*")

# ============================================================
# CONFIG
# ============================================================
MAX_DISPLAY_ROWS = 3000
YEAR_LEVELS = ["1st Year", "2nd Year", "3rd Year", "4th Year", "5th Year"]
PASSING_GRADE = 3.00
DEANS_LIST_THRESHOLD = 1.75
DEANS_LIST_MAX_GRADE = 2.50


def norm(s):
    """Normalize a header for matching: lowercase letters/digits only."""
    return re.sub(r"[^a-z0-9]", "", str(s).lower())


ALIASES = {
    "student_no":   ["student no", "student number", "student id", "id no", "id number", "id"],
    "student":      ["student name", "student", "full name", "name", "learner"],
    "program":      ["program", "course", "degree", "program course"],
    "subject_code": ["subject code", "course code", "code"],
    "subject":      ["subject name", "subject", "subject title", "subject description",
                     "descriptive title", "course title", "course name", "subject code name"],
    "teacher":      ["teacher", "instructor", "professor", "faculty"],
    "units":        ["units", "unit", "credit units", "credits", "credit", "no of units"],
    "grade":        ["final grade", "grade", "final rating", "rating"],
    "remark":       ["remarks", "remark", "status"],
}
KNOWN_HEADERS = {norm(a) for names in ALIASES.values() for a in names}

# State
current_file = {"name": None, "sheets": {}, "sheet": None, "df": None, "cols": None,
                "students": {}, "term": ""}
selected = {"key": "", "miss": ""}
table_state = {"search": ""}
student_info = {"name": None, "student_no": None, "course": None, "year_level": YEAR_LEVELS[0]}
_guard = {"selector": False, "sheet": False, "info": False, "filters": False}

# ============================================================
# GREEN & GOLD THEME CSS
# ============================================================
display(HTML("""
<style>
:root {
  --ink: #031a12; --text: #eefaf2; --muted: #a8c9b6;
  --violet: #14924f;   /* primary green  (old name kept so other rules still work) */
  --cyan: #f5c542;     /* gold accent    (old name kept) */
  --rose: #fb7185; --mint: #86efac; --amber: #fcd34d;
  --gold: #f5c542; --gold-deep: #d99a0b; --forest: #0a3d2a;
  --glass-border: rgba(245,197,66,.20);
}

/* ---------- Root canvas ---------- */
.gl-root {
  position: relative; width: 100%; max-width: 1180px; margin: 8px auto; padding: 28px; box-sizing: border-box;
  border-radius: 28px; overflow: hidden; color: var(--text);
  font-family: 'Inter', 'Segoe UI', system-ui, -apple-system, sans-serif;
  background:
    radial-gradient(700px 440px at 4% -6%,  rgba(22,163,74,.50), transparent 62%),
    radial-gradient(560px 420px at 100% 4%, rgba(245,197,66,.20), transparent 62%),
    radial-gradient(760px 560px at 60% 115%, rgba(16,120,70,.40), transparent 62%),
    linear-gradient(160deg, #04251a 0%, #031a12 55%, #021109 100%);
  box-shadow: 0 30px 70px rgba(1,10,6,.65);
}
.gl-root .widget-html, .gl-root .widget-html-content, .gl-root .widget-label { color: var(--text) !important; overflow: visible !important; }
.gl-root, .gl-root * { box-sizing: border-box; }
.gl-root .widget-box, .gl-root .widget-vbox, .gl-root .widget-hbox, .gl-root .widget-html { overflow: visible !important; }
.gl-root .widget-html { margin: 0 !important; }
.gl-root .widget-hbox { gap: 8px; }
.gl-root .widget-vbox.glass { gap: 14px; }
.gl-root .col-gap { gap: 22px; }
.gl-root .gl-header { margin-bottom: 24px; }

/* ---------- Glass surfaces ---------- */
.glass {
  background: linear-gradient(145deg, rgba(20,110,70,.28), rgba(4,40,26,.38));
  border: 1px solid var(--glass-border); border-radius: 22px; padding: 20px;
  -webkit-backdrop-filter: blur(20px) saturate(150%); backdrop-filter: blur(20px) saturate(150%);
  box-shadow: 0 12px 32px rgba(1,10,6,.5), inset 0 1px 0 rgba(255,255,255,.14);
}
.glass-soft { background: rgba(255,255,255,.06); border: 1px solid rgba(245,197,66,.16); border-radius: 14px; }
.sec-title { font-size: 14px; font-weight: 700; color: var(--gold); margin: 0 0 4px 0; letter-spacing: .3px; }
.sec-sub { font-size: 11.5px; color: var(--muted); }

/* ---------- Header ---------- */
.gl-header { display: flex; align-items: center; gap: 16px; padding: 18px 22px; border-radius: 24px; }
.gl-logo {
  width: 46px; height: 46px; border-radius: 50%; display: grid; place-items: center; font-size: 22px;
  background: linear-gradient(135deg, var(--gold), var(--gold-deep)); box-shadow: 0 8px 20px rgba(245,197,66,.35);
}
.gl-eyebrow { font-size: 10.5px; font-weight: 700; letter-spacing: 3px; color: var(--gold); margin-bottom: 2px; }
.gl-title { font-family: Georgia, 'Playfair Display', 'Times New Roman', serif; font-size: 23px; font-weight: 800; color: #fff; line-height: 1.15; letter-spacing: .5px; }
.gl-title span { color: var(--gold); }
.gl-sub { font-size: 12px; color: var(--muted); margin-top: 2px; }

/* ---------- Inputs ---------- */
.gl-input input, .gl-input select, .gl-root .widget-dropdown select, .gl-root .widget-text input {
  width: 100% !important; height: 36px !important; padding: 0 12px !important; border-radius: 12px !important;
  background: rgba(255,255,255,.08) !important; color: #fff !important; font-size: 13px !important; outline: none !important;
  border: 1px solid rgba(245,197,66,.25) !important;
}
.gl-input select option { color: #0b2a1c; background: #fff; }
.gl-input input::placeholder { color: rgba(200,230,212,.55); }
.gl-input input:focus, .gl-input select:focus { border-color: var(--gold) !important; box-shadow: 0 0 0 3px rgba(245,197,66,.22) !important; }
.field-label { font-size: 11.5px; font-weight: 600; color: var(--muted); margin: 8px 0 4px 2px; display: block; }

/* ---------- Buttons ---------- */
.gl-root button.jupyter-button, .gl-root .jupyter-widgets.widget-button {
  border: 1px solid rgba(245,197,66,.28) !important; border-radius: 999px !important; font-weight: 700 !important;
  font-size: 13px !important; color: #fff !important; background: rgba(255,255,255,.08) !important;
  transition: transform .15s ease, background .15s ease, box-shadow .15s ease !important; box-shadow: none !important;
}
.gl-root button.jupyter-button:hover { background: rgba(245,197,66,.16) !important; transform: translateY(-1px); }
.gl-root button.jupyter-button:focus-visible { outline: 2px solid var(--gold) !important; outline-offset: 2px; }
/* Primary = gold pill with dark green text (like "GET STARTED") */
.gl-root .btn-primary.jupyter-button, .gl-root .btn-primary button.jupyter-button {
  background: linear-gradient(135deg, #ffe08a, var(--gold) 55%, var(--gold-deep)) !important; border-color: rgba(255,255,255,.4) !important;
  color: #0a2a1b !important; box-shadow: 0 8px 22px rgba(245,197,66,.35) !important;
}
.gl-root .btn-primary.jupyter-button:hover, .gl-root .btn-primary button.jupyter-button:hover { filter: brightness(1.08); background: linear-gradient(135deg, #ffe08a, var(--gold) 55%, var(--gold-deep)) !important; }
.gl-root .btn-icon.jupyter-button, .gl-root .btn-icon button.jupyter-button { padding: 0 !important; font-size: 18px !important; }
.gl-root .btn-wide.jupyter-button { text-align: left !important; padding: 0 14px !important; border-radius: 12px !important; }

/* ---------- Segmented tabs ---------- */
.gl-root .seg { gap: 6px; border-radius: 999px; background: rgba(0,0,0,.28); border: 1px solid rgba(245,197,66,.22);
  -webkit-backdrop-filter: blur(16px); backdrop-filter: blur(16px); box-shadow: inset 0 1px 0 rgba(255,255,255,.08); }
.gl-root .seg button.jupyter-button.seg-btn { flex: 1 1 0; min-width: 0; margin: 0 !important; height: 40px !important; border-radius: 999px !important;
  background: transparent !important; border: 1px solid transparent !important; color: var(--muted) !important; box-shadow: none !important; transform: none !important; }
.gl-root .seg button.jupyter-button.seg-btn:hover { color: #fff !important; background: rgba(255,255,255,.07) !important; }
.gl-root .seg button.jupyter-button.seg-btn.seg-active, .gl-root .seg button.jupyter-button.seg-btn.seg-active:hover {
  background: linear-gradient(135deg, #ffe08a, var(--gold) 55%, var(--gold-deep)) !important; color: #0a2a1b !important;
  border-color: rgba(255,255,255,.35) !important; box-shadow: 0 6px 18px rgba(245,197,66,.35) !important; }

/* ---------- Status pills ---------- */
.status-msg { margin: 0; padding: 9px 13px; border-radius: 12px; font-weight: 600; font-size: 12px; line-height: 1.4; }
.status-success { background: rgba(134,239,172,.14); color: #bbf7d0; border: 1px solid rgba(134,239,172,.35); }
.status-error   { background: rgba(251,113,133,.16); color: #ffc2cb; border: 1px solid rgba(251,113,133,.4); }
.status-warn    { background: rgba(245,197,66,.14);  color: #fde9a2; border: 1px solid rgba(245,197,66,.38); }

/* ---------- Profile ---------- */
.profile { display: flex; align-items: center; gap: 12px; }
.avatar { flex: 0 0 auto; width: 46px; height: 46px; border-radius: 50%; display: grid; place-items: center; font-weight: 800; font-size: 16px; color: #0a2a1b;
  background: linear-gradient(135deg, #ffe08a, var(--gold-deep)); box-shadow: 0 6px 18px rgba(245,197,66,.35); }
.profile-name { font-size: 14px; font-weight: 700; color: #fff; }
.profile-meta { font-size: 11.5px; color: var(--muted); line-height: 1.5; }
.chips { display: flex; flex-wrap: wrap; gap: 6px; }
.chip { padding: 4px 10px; border-radius: 999px; font-size: 11px; color: var(--text); background: rgba(255,255,255,.08); border: 1px solid rgba(245,197,66,.2); }
.chip b { color: var(--gold); }

/* ---------- KPI tiles ---------- */
.kpis-wrap { container-type: inline-size; margin-bottom: 20px; }
.kpis { display: grid; grid-template-columns: repeat(7, minmax(0, 1fr)); gap: 10px; align-items: stretch; }
.kpi.hero { grid-column: span 1; }
.kpi.hero .kpi-v { font-size: 24px; }
@media (max-width: 900px) { .kpis { grid-template-columns: repeat(4, minmax(0, 1fr)); } }
@media (max-width: 560px) { .kpis { grid-template-columns: repeat(2, minmax(0, 1fr)); } }
.kpi { padding: 14px 10px; border-radius: 16px; background: rgba(255,255,255,.06); border: 1px solid rgba(245,197,66,.16); text-align: center; display: flex; flex-direction: column; justify-content: center; align-items: center; min-width: 0; }
.kpi-l { font-size: 11.5px; color: var(--muted); text-align: center; }
.kpi-v { font-size: 24px; font-weight: 800; color: #fff; margin-top: 5px; line-height: 1.15; text-align: center; }
.kpi.hero { background: linear-gradient(135deg, rgba(20,146,79,.55), rgba(245,197,66,.28)); border-color: rgba(245,197,66,.55); }
.kpi.hero .kpi-v { color: var(--gold); }
.kpi.good .kpi-v { color: var(--mint); } .kpi.bad .kpi-v { color: var(--rose); }

/* ---------- Tables ---------- */
.data-table-wrap { max-height: 440px; overflow: auto; border-radius: 14px; border: 1px solid rgba(245,197,66,.2); background: rgba(2,20,13,.45); }
table.data-table { border-collapse: collapse; width: 100%; font-size: 12.5px; color: var(--text); }
table.data-table thead th { position: sticky; top: 0; z-index: 1; text-align: left; padding: 10px 12px; font-weight: 700; color: var(--gold); white-space: nowrap;
  background: rgba(6,56,38,.92); border-bottom: 1px solid rgba(245,197,66,.35); }
table.data-table tbody td { padding: 8px 12px; border-bottom: 1px solid rgba(255,255,255,.06); white-space: nowrap; }
.gl-root table.data-table { table-layout: auto !important; width: 100% !important; min-width: max-content; max-width: none !important;
  margin: 0 !important; font-family: 'Inter', 'Segoe UI', system-ui, sans-serif !important; font-size: 12.5px !important; color: var(--text) !important; background: transparent !important; }
.gl-root table.data-table, .gl-root table.data-table thead, .gl-root table.data-table tbody, .gl-root table.data-table tr,
.gl-root table.data-table tbody tr:nth-child(odd), .gl-root table.data-table tbody tr:nth-child(even), .gl-root table.data-table thead tr {
  background: transparent !important; font-family: 'Inter', 'Segoe UI', system-ui, sans-serif !important; }
.gl-root .glass, .gl-root .widget-vbox, .gl-root .widget-hbox, .gl-root .widget-html, .gl-root .widget-html-content { min-width: 0 !important; }
.gl-root .widget-html-content { max-width: 100%; }
.gl-root .data-table-wrap { display: block; max-width: 100%; overflow: auto !important; }
.gl-root table.data-table th, .gl-root table.data-table td { font-family: 'Inter', 'Segoe UI', system-ui, sans-serif !important; width: auto !important; max-width: none !important; overflow: visible !important; text-overflow: clip !important;
  text-align: left; vertical-align: middle; border-color: rgba(255,255,255,.07) !important; }
.gl-root table.data-table thead th { background: rgba(6,56,38,.96) !important; color: var(--gold) !important; font-weight: 700 !important; border-bottom: 1px solid rgba(245,197,66,.35) !important; }
.gl-root table.data-table tbody tr td { background: transparent !important; color: var(--text) !important; }
.gl-root table.data-table tbody tr:nth-child(even) td { background: rgba(255,255,255,.04) !important; }
.gl-root table.data-table tbody tr:hover td { background: rgba(245,197,66,.14) !important; }
.gl-root table.data-table tbody tr.row-selected td { background: rgba(245,197,66,.22) !important; color: #fff !important; font-weight: 700; }
.gl-root table.data-table td, .gl-root table.data-table th { white-space: nowrap; padding: 9px 16px !important; }
.gl-root table.data-table.fit { min-width: 0; }
.gl-root table.data-table.fit td, .gl-root table.data-table.fit th { white-space: normal; padding: 9px 10px !important; }
table.data-table.fit { table-layout: auto; }
table.data-table.fit th, table.data-table.fit td { white-space: normal; padding: 8px 9px; }
table.data-table.fit .sub { display: block; font-size: 11px; color: var(--muted); font-weight: 400; margin-top: 1px; }
table.data-table.fit td.name { min-width: 170px; }
.pill { display: inline-block; padding: 2px 10px; border-radius: 999px; font-size: 11px; font-weight: 700; }
.pill-pass { background: rgba(134,239,172,.18); color: #bbf7d0; }
.pill-fail { background: rgba(251,113,133,.2); color: #fecdd3; }
.pill-other { background: rgba(245,197,66,.2); color: #fde68a; }
.pill-none { color: var(--muted); }
.empty { text-align: center; padding: 34px 10px; }
.empty .big { font-size: 34px; } .empty .t { font-size: 14px; font-weight: 700; margin-top: 4px; } .empty .s { font-size: 12px; color: var(--muted); margin-top: 4px; }

::-webkit-scrollbar { width: 7px; height: 7px; }
::-webkit-scrollbar-thumb { background: rgba(245,197,66,.35); border-radius: 10px; }
::-webkit-scrollbar-track { background: transparent; }
@media (prefers-reduced-motion: reduce) { .gl-root * { transition: none !important; } }
</style>
"""))


# ============================================================
# FILE PARSING
# ============================================================

def raw_from_csv(data):
    try:
        text = data.decode("utf-8-sig")
    except UnicodeDecodeError:
        text = data.decode("latin-1")
    rows = list(csv.reader(io.StringIO(text)))
    rows = [[(c.strip() if c.strip() else None) for c in r] for r in rows]
    width = max((len(r) for r in rows), default=0)
    rows = [r + [None] * (width - len(r)) for r in rows]
    return pd.DataFrame(rows)


def find_header_row(raw):
    best_idx, best_score = 0, 0
    for i in range(min(len(raw), 30)):
        score = sum(1 for v in raw.iloc[i].tolist() if pd.notna(v) and norm(v) in KNOWN_HEADERS)
        if score > best_score:
            best_idx, best_score = i, score
    return (best_idx if best_score >= 2 else 0), best_score


def frame_from_raw(raw):
    raw = raw.dropna(how="all").dropna(axis=1, how="all").reset_index(drop=True)
    if raw.empty:
        return pd.DataFrame(), 0
    idx, score = find_header_row(raw)
    headers, seen = [], {}
    for j, v in enumerate(raw.iloc[idx].tolist()):
        name = str(v).strip() if pd.notna(v) and str(v).strip() else f"Column {j + 1}"
        seen[name] = seen.get(name, 0) + 1
        headers.append(name if seen[name] == 1 else f"{name} ({seen[name]})")
    df = raw.iloc[idx + 1:].copy()
    df.columns = headers
    df = df.dropna(how="all").reset_index(drop=True)
    return df.infer_objects(), score


TERM_RE = re.compile(r"\b(?:1st|2nd|3rd|first|second|third)\s+(?:semester|sem|term|trimester)\b|\bsummer\b|\bmidyear\b", re.I)
SY_RE = re.compile(r"(?:S\.?\s?Y\.?|school\s+year|academic\s+year|A\.?Y\.?)\s*:?\s*(\d{4}\s*[-–/]\s*\d{2,4})", re.I)


def detect_term(raws):
    term, sy = "", ""
    for raw in raws.values():
        for v in raw.head(6).to_numpy().ravel():
            if not isinstance(v, str):
                continue
            m = TERM_RE.search(v)
            if m and not term:
                term = " ".join(w.capitalize() for w in m.group(0).split())
            s = SY_RE.search(v)
            if s and not sy:
                sy = "S.Y. " + re.sub(r"\s+", "", s.group(1))
    return ", ".join(p for p in (term, sy) if p)


def find_column(df, aliases):
    lookup = {}
    for c in df.columns:
        lookup.setdefault(norm(c), c)
    for a in aliases:
        if norm(a) in lookup:
            return lookup[norm(a)]
    return None


def detect_columns(df):
    if df is None or df.empty:
        return None
    cols = {k: find_column(df, v) for k, v in ALIASES.items()}
    if not (cols["units"] and cols["grade"] and (cols["subject"] or cols["subject_code"])):
        return None
    return cols


def parse_upload(name, data):
    lower = name.lower()
    if lower.endswith((".xlsx", ".xlsm", ".xls")):
        raws = pd.read_excel(io.BytesIO(data), sheet_name=None, header=None)
    elif lower.endswith(".csv"):
        # Headerless 7-column CSV: Student Name, Program, Subject Code, Subject Name, Units, Teacher, Grade
        raw = raw_from_csv(data)
        _, header_score = find_header_row(raw)
        if header_score < 2 and raw.shape[1] == 7:
            header = pd.DataFrame([["Student Name", "Program", "Subject Code", "Subject Name",
                                    "Units", "Teacher", "Grade"]])
            raw = pd.concat([header, raw], ignore_index=True)
        raws = {"CSV": raw}
    else:
        raise ValueError("Unsupported file type. Please upload a .csv, .xlsx, or .xls file.")

    sheets = {}
    for sheet_name, raw in raws.items():
        df, score = frame_from_raw(raw)
        sheets[sheet_name] = {"df": df, "cols": detect_columns(df), "score": score}
    return sheets, detect_term(raws)


def choose_sheet(sheets):
    """Prefer the detailed Records sheet over a calculated Summary sheet."""
    good = [n for n, sh in sheets.items() if sh["cols"] is not None]
    if not good:
        return max(sheets, key=lambda n: (sheets[n]["score"], len(sheets[n]["df"])))
    for preferred in ("records", "grade records", "grades", "student records"):
        for name in good:
            if norm(name) == norm(preferred):
                return name
    return max(good, key=lambda n: len(sheets[n]["df"]))


# ============================================================
# GRADE LOGIC
# ============================================================

def to_float(v):
    if v is None or (not isinstance(v, str) and pd.isna(v)):
        return None
    try:
        f = float(str(v).strip())
    except ValueError:
        return None
    return None if pd.isna(f) else f


def is_blank(v):
    return v is None or (isinstance(v, str) and not v.strip()) or (not isinstance(v, str) and pd.isna(v))


def fmt_grade(v):
    g = to_float(v)
    if g is not None:
        return f"{g:.2f}"
    return "" if is_blank(v) else str(v).strip()


def grade_quality(v):
    """Descriptive evaluation using the Philippine 1.00–5.00 grading scale."""
    if is_blank(v):
        return "No grade"
    if str(v).strip().upper() in ("INC", "INCOMPLETE"):
        return "Incomplete"
    g = to_float(v)
    if g is None:
        return "Not rated"
    if g <= 1.25:
        return "Excellent"
    if g <= 1.75:
        return "Very Good"
    if g <= 2.25:
        return "Good"
    if g <= 2.75:
        return "Satisfactory"
    if g <= 3.00:
        return "Passed"
    if g <= 4.00:
        return "Conditional"
    return "Failed"


def fmt_num(x):
    return "—" if x is None else f"{x:.2f}"


def compute_grade_remark(v):
    if is_blank(v):
        return ""
    if str(v).strip().upper() in ("INC", "INCOMPLETE"):
        return "Incomplete"
    g = to_float(v)
    if g is None:
        return ""
    if g <= PASSING_GRADE:
        return "Passed"
    if g == 4.00:
        return "Conditional"
    return "Failed"


def clean_str(series):
    return series.where(series.notna(), "").astype(str).str.strip()


def row_keys(df, cols):
    n = len(df)
    empty = pd.Series([""] * n, index=df.index)
    no = clean_str(df[cols["student_no"]]).str.replace(r"\.0$", "", regex=True) if cols["student_no"] else empty
    nm = clean_str(df[cols["student"]]) if cols["student"] else empty
    keys = [a if a else (("name:" + b.lower()) if b else "") for a, b in zip(no, nm)]
    return pd.Series(keys, index=df.index)


def list_students(df, cols):
    if not cols or not (cols["student_no"] or cols["student"]):
        return {}
    keys = row_keys(df, cols)
    out = {}
    for idx, key in keys.items():
        if not key or key in out:
            continue
        row = df.loc[idx]
        if is_blank(row[cols["units"]]) and is_blank(row[cols["grade"]]):
            continue
        name = "" if not cols["student"] or is_blank(row[cols["student"]]) else str(row[cols["student"]]).strip()
        no = "" if not cols["student_no"] or is_blank(row[cols["student_no"]]) else re.sub(r"\.0$", "", str(row[cols["student_no"]]).strip())
        prog = "" if not cols["program"] or is_blank(row[cols["program"]]) else str(row[cols["program"]]).strip()
        out[key] = {"key": key, "name": name or no or key, "student_no": no, "program": prog}
    return out


def summarize(subset, cols):
    rows, total_units, weighted, counted, grades, failed = [], 0.0, 0.0, 0.0, [], 0
    incomplete = 0
    for _, r in subset.iterrows():
        units = to_float(r[cols["units"]])
        grade_raw = r[cols["grade"]]
        if units is None and is_blank(grade_raw):
            continue
        units = units or 0.0
        g = to_float(grade_raw)

        remark = ""
        if cols["remark"] and not is_blank(r[cols["remark"]]):
            remark = str(r[cols["remark"]]).strip()
        else:
            remark = compute_grade_remark(grade_raw)

        def text(col):
            return "" if not col or is_blank(r[col]) else str(r[col]).strip()

        if remark.lower() == "incomplete":
            incomplete += 1
        total_units += units
        if g is not None:
            weighted += g * units
            counted += units
            grades.append(g)
        if remark.lower() == "failed" or (g is not None and g > PASSING_GRADE):
            failed += 1
        rows.append({"code": text(cols["subject_code"]), "subject": text(cols["subject"]),
                     "teacher": text(cols["teacher"]), "units": units, "grade": grade_raw,
                     "quality": grade_quality(grade_raw), "remark": remark})

    gwa = (weighted / counted) if counted else None
    best, worst = (min(grades), max(grades)) if grades else (None, None)
    dean = (gwa is not None and round(gwa, 2) <= DEANS_LIST_THRESHOLD
            and worst is not None and worst <= DEANS_LIST_MAX_GRADE)
    return {"rows": rows, "subjects": len(rows), "total_units": total_units, "gwa": gwa,
            "best": best, "worst": worst, "failed": failed, "incomplete": incomplete, "dean_list": dean,
            "has_code": any(x["code"] for x in rows), "has_teacher": any(x["teacher"] for x in rows)}


def student_subset(key):
    df, cols = current_file["df"], current_file["cols"]
    return df[row_keys(df, cols) == key]


def class_summary():
    df, cols = current_file["df"], current_file["cols"]
    keys = row_keys(df, cols)
    out = []
    for key, info in current_file["students"].items():
        s = summarize(df[keys == key], cols)
        out.append({**info, **s})
    return sorted(out, key=lambda x: x["name"].lower())


def evaluation(c):
    """One overall label per student (priority: failed > incomplete > Dean's Lister > passed)."""
    if c["failed"]:
        return "Has failed subject(s)"
    if c["incomplete"]:
        return "Has incomplete"
    if c["dean_list"]:
        return "Dean's Lister"
    return "Passed"


SORT_OPTIONS = [("Name A → Z", "az"), ("Name Z → A", "za"),
                ("GWA: best → lowest", "gwa_best"), ("GWA: lowest → best", "gwa_low")]
EVAL_OPTIONS = [("All students", "all"), ("Dean's Lister", "dean"), ("No failed subjects", "clean"),
                ("Has failed subject(s)", "failed"), ("Has incomplete", "inc")]


def class_view(sort="az", initial="All", ev="all"):
    """The class summary after the sort / initial / evaluation filters are applied."""
    rows = class_summary()
    if initial != "All":
        rows = [c for c in rows if c["name"][:1].upper() == initial]
    keep = {"all": lambda c: True, "dean": lambda c: c["dean_list"], "clean": lambda c: not c["failed"],
            "failed": lambda c: c["failed"] > 0, "inc": lambda c: c["incomplete"] > 0}[ev]
    rows = [c for c in rows if keep(c)]
    if sort == "za":
        rows.sort(key=lambda c: c["name"].lower(), reverse=True)
    elif sort in ("gwa_best", "gwa_low"):
        have = sorted([c for c in rows if c["gwa"] is not None], key=lambda c: c["gwa"], reverse=(sort == "gwa_low"))
        rows = have + [c for c in rows if c["gwa"] is None]
    return rows


def match_student(typed, students):
    t = norm(typed or "")
    if not t or not students:
        return None
    exact = [k for k, s in students.items() if norm(s["name"]) == t]
    if len(exact) == 1:
        return exact[0]
    words = [w for w in re.split(r"\s+", (typed or "").lower().strip()) if w]
    sub = [k for k, s in students.items() if words and all(w in s["name"].lower() for w in words)]
    if len(sub) == 1:
        return sub[0]
    sub = [k for k, s in students.items() if t in norm(s["name"])]
    return sub[0] if len(sub) == 1 else None


# ============================================================
# UI WIDGETS  (sidebar + tabbed main panel)
# ============================================================

def set_visible(w, visible):
    w.layout.display = None if visible else "none"


def labeled(label_text, widget):
    lbl = widgets.HTML(f'<span class="field-label">{label_text}</span>')
    return widgets.VBox([lbl, widget], layout=widgets.Layout(width="100%"))


def glass(children, **layout):
    layout.setdefault("gap", "10px")
    layout.setdefault("overflow", "visible")
    box = widgets.VBox(children, layout=widgets.Layout(**layout))
    box.add_class("glass")
    return box


def btn(text, width="auto", primary=False, height="36px"):
    b = widgets.Button(description=text, layout=widgets.Layout(width=width, height=height))
    b.add_class("btn-primary" if primary else "btn-soft")
    return b


# ---- Header ----
header = widgets.HTML("""
<div class="glass gl-header">
  <div class="gl-logo">🎓</div>
  <div>
    <div class="gl-eyebrow">NOTRE DAME OF MARBEL UNIVERSITY</div>
    <div class="gl-title">Student <span>Grade</span> Report</div>
    <div class="gl-sub">Upload a grade file, pick a student, download a PDF report.</div>
  </div>
</div>""", layout=widgets.Layout(width="100%"))

# ---- Sidebar: upload ----
uploader = widgets.FileUpload(accept=".csv,.xlsx,.xls", multiple=False, description="Choose file",
                              layout=widgets.Layout(width="100%", height="38px"))
uploader.add_class("btn-primary")
dd_sheet = widgets.Dropdown(options=[], layout=widgets.Layout(width="100%"))
dd_sheet.add_class("gl-input")
sheet_box = labeled("Sheet", dd_sheet)
set_visible(sheet_box, False)
status = widgets.HTML()
file_area = widgets.HTML()
upload_card = glass([widgets.HTML('<div class="sec-title">Upload grades</div>'),
                     uploader, status, file_area, sheet_box])

# ---- Sidebar: student picker ----
dd_student = widgets.Dropdown(options=[("Select a student", "")], layout=widgets.Layout(flex="1 1 auto", min_width="0", width="auto"))
dd_student.add_class("gl-input")
btn_prev = btn("‹", "36px"); btn_prev.add_class("btn-icon"); btn_prev.tooltip = "Previous student"
btn_next = btn("›", "36px"); btn_next.add_class("btn-icon"); btn_next.tooltip = "Next student"
picker_row = widgets.HBox([btn_prev, dd_student, btn_next], layout=widgets.Layout(gap="6px", width="100%", align_items="center", overflow="visible"))
profile_area = widgets.HTML()
selector_card = glass([widgets.HTML('<div class="sec-title">Student</div>'), picker_row, profile_area])
set_visible(selector_card, False)

# ---- Sidebar: report details (optional, used in the printed report) ----
txt_student_name = widgets.Text(placeholder="Type a name, press Enter", continuous_update=False, layout=widgets.Layout(width="100%"))
txt_course = widgets.Text(placeholder="e.g. BS Information Technology", continuous_update=False, layout=widgets.Layout(width="100%"))
dd_year_level = widgets.Dropdown(options=YEAR_LEVELS, value=YEAR_LEVELS[0], layout=widgets.Layout(width="100%"))
for w in (txt_student_name, txt_course, dd_year_level):
    w.add_class("gl-input")
details_body = widgets.VBox([labeled("Full name", txt_student_name), labeled("Program", txt_course), labeled("Year level", dd_year_level)])
set_visible(details_body, False)
btn_details = btn("Report details  ▾", "100%")
btn_details.add_class("btn-wide")


def toggle_details(e):
    opened = details_body.layout.display == "none"
    set_visible(details_body, opened)
    btn_details.description = "Report details  " + ("▴" if opened else "▾")


btn_details.on_click(toggle_details)
details_card = glass([btn_details, details_body])

sidebar = widgets.VBox([upload_card, selector_card, details_card],
                       layout=widgets.Layout(flex="0 0 310px", width="310px", min_width="260px", overflow="visible"))

# ---- Main: tabs ----
TABS = ["Student report", "Class summary", "Data preview"]
class SegControl:
    """Equal-width tab bar built from plain buttons (reliable across ipywidgets versions)."""

    def __init__(self, options):
        self.options, self._value = list(options), options[0]
        self.buttons = [widgets.Button(description=o, layout=widgets.Layout(flex="1 1 0", width="auto", height="40px", margin="0"))
                        for o in self.options]
        for b in self.buttons:
            b.add_class("seg-btn")
            b.on_click(lambda e, name=b.description: setattr(self, "value", name))
        self.box = widgets.HBox(self.buttons, layout=widgets.Layout(width="100%", padding="5px", overflow="visible"))
        self.box.add_class("seg")
        self._sync()

    def _sync(self):
        for b in self.buttons:
            (b.add_class if b.description == self._value else b.remove_class)("seg-active")

    @property
    def value(self):
        return self._value

    @value.setter
    def value(self, v):
        self._value = v
        self._sync()
        show_tab(v)


seg = SegControl(TABS)

# Student report panel
grade_summary_area = widgets.HTML()
report_status = widgets.HTML()
btn_download_report = btn("📄 Download report (PDF)", "220px", primary=True)
report_actions_row = widgets.HBox([report_status, btn_download_report],
                                  layout=widgets.Layout(justify_content="flex-end", align_items="center", margin="18px 0 0 0"))
set_visible(report_actions_row, False)
grade_summary_card = glass([grade_summary_area, report_actions_row])

# Class summary panel
class_summary_area = widgets.HTML()
class_status = widgets.HTML()
btn_download_class = btn("⬇ Class summary (PDF)", "210px")
class_actions_row = widgets.HBox([class_status, btn_download_class],
                                 layout=widgets.Layout(justify_content="flex-end", align_items="center", margin="18px 0 0 0"))
dd_sort = widgets.Dropdown(options=SORT_OPTIONS, value="az", layout=widgets.Layout(width="100%"))
dd_initial = widgets.Dropdown(options=["All"], value="All", layout=widgets.Layout(width="100%"))
dd_eval = widgets.Dropdown(options=EVAL_OPTIONS, value="all", layout=widgets.Layout(width="100%"))
for _w in (dd_sort, dd_initial, dd_eval):
    _w.add_class("gl-input")
btn_reset_filters = btn("Reset", "80px")
filter_cells = [widgets.VBox([widgets.HTML(f'<span class="field-label" style="margin-top:0;">{t}</span>'), w],
                             layout=widgets.Layout(flex="1 1 150px", min_width="140px", overflow="visible"))
                for t, w in (("Sort by", dd_sort), ("Starts with", dd_initial), ("Evaluation", dd_eval))]
filter_row = widgets.HBox(filter_cells + [widgets.VBox([widgets.HTML('<span class="field-label" style="margin-top:0;">&nbsp;</span>'), btn_reset_filters],
                                                      layout=widgets.Layout(flex="0 0 auto", overflow="visible"))],
                          layout=widgets.Layout(width="100%", flex_flow="row wrap", align_items="flex-end", overflow="visible"))
class_summary_card = glass([filter_row, class_summary_area, class_actions_row])
set_visible(class_summary_card, False)

# Data preview panel
search_box = widgets.Text(placeholder="🔍 Filter rows (searches every column)", layout=widgets.Layout(flex="1 1 auto", min_width="0", width="auto"))
search_box.add_class("gl-input")
btn_download = btn("⬇ Filtered data (PDF)", "190px")
table_area = widgets.HTML()
controls_row = widgets.HBox([search_box, btn_download], layout=widgets.Layout(gap="10px", margin="0 0 8px 0", align_items="center"))
preview_card = glass([controls_row, table_area])
set_visible(preview_card, False)

PANELS = {TABS[0]: grade_summary_card, TABS[1]: class_summary_card, TABS[2]: preview_card}
main_panel = widgets.VBox([seg.box, grade_summary_card, class_summary_card, preview_card],
                          layout=widgets.Layout(flex="1 1 520px", min_width="0", gap="14px", overflow="visible"))

body = widgets.HBox([sidebar, main_panel],
                    layout=widgets.Layout(width="100%", flex_flow="row wrap", align_items="flex-start", gap="14px", overflow="visible"))
app_container = widgets.VBox([header, body], layout=widgets.Layout(width="100%"))
app_container.add_class("gl-root")
for _w in (sidebar, main_panel, body):
    _w.add_class("col-gap")


def show_tab(name):
    for t, panel in PANELS.items():
        set_visible(panel, t == name)




# ============================================================
# RENDERING
# ============================================================

def esc(v):
    return html_lib.escape(str(v))


def initials(name):
    parts = [p for p in re.split(r"\s+", (name or "").strip()) if p]
    return (parts[0][0] + (parts[-1][0] if len(parts) > 1 else "")).upper() if parts else "?"


def remark_html(r):
    if not r:
        return '<span class="pill-none">—</span>'
    cls = {"passed": "pill-pass", "failed": "pill-fail"}.get(r.lower(), "pill-other")
    return f'<span class="pill {cls}">{esc(r)}</span>'


def empty_html(icon, title, sub=""):
    s = f'<div class="s">{sub}</div>' if sub else ""
    return f'<div class="empty"><div class="big">{icon}</div><div class="t">{title}</div>{s}</div>'


def render_profile():
    if not current_file["students"]:
        profile_area.value = ""
        return
    if not selected["key"]:
        profile_area.value = f'<div class="status-msg status-warn" style="margin-top:4px;">{esc("Select a student to see their grades.")}</div>'
        return
    meta = [v for v in (student_info["student_no"], student_info["course"], student_info["year_level"], current_file["term"]) if v]
    meta_html = "<br>".join(esc(m) for m in meta)
    profile_area.value = (f'<div class="profile glass-soft" style="padding:14px 16px; margin-top:4px;">'
                          f'<div class="avatar">{esc(initials(student_info["name"]))}</div>'
                          f'<div><div class="profile-name">{esc(student_info["name"] or "")}</div>'
                          f'<div class="profile-meta">{meta_html}</div></div></div>')


def render_stats():
    df = current_file["df"]
    if df is None:
        file_area.value = ""
        return
    chips = [f'<span class="chip"><b>{esc(current_file["name"])}</b></span>',
             f'<span class="chip"><b>{len(df):,}</b> rows</span>']
    if current_file["students"]:
        chips.append(f'<span class="chip"><b>{len(current_file["students"])}</b> students</span>')
    if current_file["term"]:
        chips.append(f'<span class="chip">{esc(current_file["term"])}</span>')
    file_area.value = f'<div class="chips" style="margin-top:2px;">{"".join(chips)}</div>'


def filtered_df():
    df = current_file["df"]
    if df is None:
        return None
    q = table_state["search"]
    if not q:
        return df
    mask = df.apply(lambda row: row.astype(str).str.contains(q, case=False, na=False, regex=False).any(), axis=1)
    return df[mask]


def render_table():
    df = filtered_df()
    if df is None:
        table_area.value = empty_html("📂", "No file loaded yet", "Choose a CSV or Excel file in the sidebar.")
        return
    if df.empty:
        table_area.value = '<div class="status-msg status-error">No rows match your search.</div>'
        return
    note, shown = "", df
    if len(df) > MAX_DISPLAY_ROWS:
        shown = df.head(MAX_DISPLAY_ROWS)
        note = (f'<div class="status-msg status-warn" style="margin:0 0 8px 0;">Showing first {MAX_DISPLAY_ROWS:,} of {len(df):,} rows. '
                f'Use the filter, or <code>current_file["df"]</code> in another cell.</div>')
    table_area.value = f'{note}<div class="data-table-wrap">{shown.to_html(index=False, classes="data-table", border=0, na_rep="")}</div>'


def grade_table_html(summary, themed=True):
    head = ["Code"] if summary["has_code"] else []
    head += ["Subject"] + (["Teacher"] if summary["has_teacher"] else []) + ["Units", "Grade", "Evaluation", "Remark"]
    ths = "".join(f"<th>{h}</th>" for h in head)
    body_rows = []
    for r in summary["rows"]:
        tds = []
        if summary["has_code"]:
            tds.append(f"<td>{esc(r['code'])}</td>")
        tds.append(f"<td>{esc(r['subject'])}</td>")
        if summary["has_teacher"]:
            tds.append(f"<td>{esc(r['teacher'])}</td>")
        tds.append(f'<td style="text-align:center;">{r["units"]:g}</td>')
        tds.append(f'<td style="text-align:center;">{esc(fmt_grade(r["grade"]))}</td>')
        tds.append(f'<td style="text-align:center;">{esc(r["quality"])}</td>')
        tds.append(f'<td style="text-align:center;">{remark_html(r["remark"]) if themed else esc(r["remark"])}</td>')
        body_rows.append("<tr>" + "".join(tds) + "</tr>")
    return ths, "".join(body_rows)


def current_summary():
    df, cols = current_file["df"], current_file["cols"]
    if df is None:
        return "none", None
    if cols is None:
        return "nocols", None
    if current_file["students"]:
        if not selected["key"]:
            miss = selected["miss"]
            msg = "Select a student in the sidebar to see their grades."
            if miss:
                msg = f"No student named '{miss}' was found in this file. {msg}"
            return "prompt", msg
        return "ok", summarize(student_subset(selected["key"]), cols)
    return "ok", summarize(df, cols)


def kpi(label, value, cls=""):
    return f'<div class="kpi {cls}"><div class="kpi-l">{label}</div><div class="kpi-v">{value}</div></div>'


def render_grade_summary():
    state, payload = current_summary()
    set_visible(report_actions_row, False)

    if state == "none":
        grade_summary_area.value = empty_html("🎓", "Upload a grade file to begin", "Your student report will appear here.")
        return
    if state == "nocols":
        found = ", ".join(str(c) for c in current_file["df"].columns)
        others = [n for n in current_file["sheets"] if n != current_file["sheet"]]
        extra = f"<br>Other sheets: {esc(', '.join(others))}" if others else ""
        grade_summary_area.value = (
            '<div class="status-msg status-warn">Subject, Units and Grade columns were not found, so no report can be built.'
            f'<br><span style="font-size:11px; opacity:.85;">Columns in "{esc(current_file["sheet"])}": {esc(found)}{extra}</span></div>')
        return
    if state == "prompt":
        grade_summary_area.value = empty_html("👈", "Pick a student", esc(payload))
        return

    s = payload
    name = student_info["name"] or ""
    if not s["rows"]:
        grade_summary_area.value = f'<div class="status-msg status-error">No grade records found for {esc(name)}.</div>'
        return

    ths, trs = grade_table_html(s)
    dean_cls, dean_txt = ("good", "Yes 🎉") if s["dean_list"] else ("", "No")
    fail_cls = "bad" if s["failed"] else ""
    grade_summary_area.value = f"""
    <div style="display:flex; justify-content:space-between; align-items:baseline; flex-wrap:wrap; gap:6px; margin-bottom:12px;">
      <div class="sec-title" style="margin:0; font-size:15px;">{esc(name)}</div>
      <div class="sec-sub">{esc(current_file['term'])}</div>
    </div>
    <div class="kpis-wrap"><div class="kpis">
      {kpi('GWA · ' + grade_quality(s['gwa']), fmt_num(s['gwa']), 'hero')}
      {kpi('Total units', f"{s['total_units']:g}")}
      {kpi('Subjects', s['subjects'])}
      {kpi('Best grade', fmt_num(s['best']))}
      {kpi('Lowest grade', fmt_num(s['worst']))}
      {kpi('Failed', s['failed'], fail_cls)}
      {kpi("Dean's Lister", dean_txt, dean_cls)}
    </div></div>
    <div class="data-table-wrap" style="max-height:360px;">
      <table class="data-table fit"><thead><tr>{ths}</tr></thead><tbody>{trs}</tbody></table>
    </div>
    <div class="sec-sub" style="margin-top:10px; text-align:center;">Dean's Lister: GWA of {DEANS_LIST_THRESHOLD:.2f} or better with no grade worse than {DEANS_LIST_MAX_GRADE:.2f}.</div>
    """
    set_visible(report_actions_row, True)


EVAL_PILL = {"Dean's Lister": "pill-pass", "Passed": "pill-pass", "Has failed subject(s)": "pill-fail", "Has incomplete": "pill-other"}


def refresh_initials():
    letters = sorted({c["name"][:1].upper() for c in current_file["students"].values() if c["name"]})
    keep = dd_initial.value if dd_initial.value in letters else "All"
    _guard["filters"] = True
    try:
        dd_initial.options = ["All"] + letters
        dd_initial.value = keep
    finally:
        _guard["filters"] = False


def reset_class_filters():
    _guard["filters"] = True
    try:
        dd_sort.value, dd_eval.value = "az", "all"
    finally:
        _guard["filters"] = False
    refresh_initials()
    dd_initial.value = "All"


def render_class_summary():
    if current_file["df"] is None or current_file["cols"] is None or not current_file["students"]:
        set_visible(class_summary_card, False)
        return
    set_visible(class_summary_card, seg.value == TABS[1])
    total = len(current_file["students"])
    view = class_view(dd_sort.value, dd_initial.value, dd_eval.value)
    ctr = 'style="text-align:center;"'
    rows = []
    for c in view:
        cls = ' class="row-selected"' if c["key"] == selected["key"] else ""
        ev = evaluation(c)
        ev_html = f'<span class="pill {EVAL_PILL[ev]}">{esc(ev)}</span>'
        fail = f'<span class="pill pill-fail">{c["failed"]}</span>' if c["failed"] else "0"
        sub = " · ".join(x for x in (c["student_no"], c["program"]) if x)
        sub_html = f'<span class="sub">{esc(sub)}</span>' if sub else ""
        rows.append(
            f'<tr{cls}><td class="name"><b>{esc(c["name"])}</b>{sub_html}</td>'
            f'<td {ctr}>{c["subjects"]}</td><td {ctr}>{c["total_units"]:g}</td>'
            f'<td {ctr}><b>{fmt_num(c["gwa"])}</b></td><td {ctr}>{fmt_num(c["best"])}</td>'
            f'<td {ctr}>{fmt_num(c["worst"])}</td><td {ctr}>{fail}</td><td {ctr}>{ev_html}</td></tr>')
    heads = ["Student", "Subjects", "Units", "GWA", "Best", "Lowest", "Failed", "Evaluation"]
    ths = "".join(f"<th>{h}</th>" for h in heads)
    head = ('<div class="sec-title" style="font-size:15px;">All students</div>'
            f'<div class="sec-sub" style="margin-bottom:2px;">Showing <b style="color:#fff;">{len(view)}</b> of {total} students. The selected student is highlighted.</div>')
    if not view:
        class_summary_area.value = head + '<div class="status-msg status-warn" style="margin-top:12px;">No students match these filters.</div>'
        return
    class_summary_area.value = (head +
        f'<div class="data-table-wrap" style="max-height:420px; margin-top:12px;"><table class="data-table fit"><thead><tr>{ths}</tr></thead>'
        f'<tbody>{"".join(rows)}</tbody></table></div>')


def on_filter_change(change):
    if _guard["filters"]:
        return
    class_status.value = ""
    render_class_summary()


for _w in (dd_sort, dd_initial, dd_eval):
    _w.observe(on_filter_change, names="value")
btn_reset_filters.on_click(lambda e: (reset_class_filters(), render_class_summary()))


def render_all():
    render_profile()
    render_stats()
    render_table()
    render_grade_summary()
    render_class_summary()
    show_tab(seg.value)
    if current_file["df"] is None or not current_file["students"]:
        set_visible(class_summary_card, False)


# ============================================================
# STUDENT SELECTION
# ============================================================

def student_label(s):
    return f'{s["name"]}  ({s["student_no"]})' if s["student_no"] else s["name"]


def refresh_selector(preferred_key=None):
    students = current_file["students"]
    _guard["selector"] = True
    try:
        opts = [("Select a student", "")] + [
            (student_label(s), k) for k, s in sorted(students.items(), key=lambda kv: kv[1]["name"].lower())]
        dd_student.options = opts
        dd_student.value = preferred_key if preferred_key in students else ""
    finally:
        _guard["selector"] = False
    set_visible(selector_card, bool(students))


def apply_selection(key):
    info = current_file["students"].get(key)
    selected.update(key=key if info else "", miss="")
    if info:
        student_info["name"] = info["name"]
        student_info["student_no"] = info["student_no"] or None
        if info["program"]:
            student_info["course"] = info["program"]
        _guard["info"] = True
        try:
            txt_student_name.value = student_info["name"]
            txt_course.value = student_info["course"] or ""
        finally:
            _guard["info"] = False


def auto_match_student():
    students = current_file["students"]
    if not students:
        selected.update(key="", miss="")
        refresh_selector(None)
        return
    key = match_student(student_info["name"], students)
    if key and key == selected["key"]:
        refresh_selector(key)
        return
    refresh_selector(key)
    if key:
        apply_selection(key)
    else:
        selected.update(key="", miss=(student_info["name"] or "").strip())


def on_student_change(change):
    if _guard["selector"]:
        return
    apply_selection(change["new"])
    report_status.value = ""
    if change["new"]:
        seg.value = TABS[0]
    render_all()


dd_student.observe(on_student_change, names="value")


def step_student(delta):
    keys = [k for k, _ in sorted(current_file["students"].items(), key=lambda kv: kv[1]["name"].lower())]
    if not keys:
        return
    i = keys.index(selected["key"]) if selected["key"] in keys else (-1 if delta > 0 else 0)
    dd_student.value = keys[(i + delta) % len(keys)]


btn_prev.on_click(lambda e: step_student(-1))
btn_next.on_click(lambda e: step_student(1))


def on_details_change(change):
    if _guard["info"]:
        return
    student_info.update(name=txt_student_name.value.strip() or None,
                        course=txt_course.value.strip() or None,
                        year_level=dd_year_level.value)
    if change["owner"] is txt_student_name:
        student_info["student_no"] = None
        auto_match_student()
    render_all()


for _w in (txt_student_name, txt_course, dd_year_level):
    _w.observe(on_details_change, names="value")


# ============================================================
# LOADING
# ============================================================

def activate_sheet(sheet_name):
    sh = current_file["sheets"][sheet_name]
    current_file.update(sheet=sheet_name, df=sh["df"], cols=sh["cols"],
                        students=list_students(sh["df"], sh["cols"]) if sh["cols"] else {})
    table_state["search"] = ""
    search_box.value = ""
    report_status.value = ""
    class_status.value = ""
    auto_match_student()
    reset_class_filters()
    seg.value = TABS[0]
    render_all()


def on_sheet_change(change):
    if _guard["sheet"]:
        return
    activate_sheet(change["new"])


dd_sheet.observe(on_sheet_change, names="value")


def load_file(name, data):
    try:
        sheets, term = parse_upload(name, data)
    except Exception as exc:
        status.value = f'<div class="status-msg status-error">Could not read \'{esc(name)}\': {esc(exc)}</div>'
        return
    if not sheets or all(s["df"].empty for s in sheets.values()):
        status.value = f'<div class="status-msg status-error">\'{esc(name)}\' has no data.</div>'
        return

    chosen = choose_sheet(sheets)
    current_file.update(name=name, sheets=sheets, term=term)

    valid_sheets = [n for n, sh in sheets.items() if sh["cols"] is not None]
    sheet_options = valid_sheets if valid_sheets else list(sheets.keys())
    if chosen not in sheet_options:
        chosen = sheet_options[0]

    _guard["sheet"] = True
    try:
        dd_sheet.options = sheet_options
        dd_sheet.value = chosen
    finally:
        _guard["sheet"] = False
    set_visible(sheet_box, len(sheet_options) > 1)

    activate_sheet(chosen)

    df = current_file["df"]
    where = f" (sheet '{chosen}')" if len(sheets) > 1 else ""
    extra = f", {len(current_file['students'])} students" if current_file["students"] else ""
    status.value = f'<div class="status-msg status-success">Loaded{esc(where)}: {len(df)} rows{extra}.</div>'


def on_upload_change(change):
    uploaded = change["new"]
    if not uploaded:
        return
    if isinstance(uploaded, dict):          # ipywidgets 7.x
        name, info = next(iter(uploaded.items()))
    else:                                   # ipywidgets 8.x
        info = uploaded[0]
        name = info["name"]
    load_file(name, bytes(info["content"]))


uploader.observe(on_upload_change, names="value")


def on_search_change(change):
    table_state["search"] = change["new"].strip()
    render_table()


search_box.observe(on_search_change, names="value")


# ============================================================
# DOWNLOADS
# ============================================================

def deliver_file(out_name, status_widget, hint=""):
    try:
        from google.colab import files as colab_files
        colab_files.download(out_name)
        status_widget.value = f'<div class="status-msg status-success">Downloading \'{esc(out_name)}\'...</div>'
    except ImportError:
        status_widget.value = (f'<div class="status-msg status-success">Saved \'{esc(out_name)}\' in the '
                               f'notebook\'s working directory.{hint}</div>')


def safe_filename(text, fallback):
    cleaned = "".join(c if c.isalnum() or c in (" ", "_", "-") else "" for c in (text or "")).strip().replace(" ", "_")
    return cleaned or fallback


def ensure_reportlab():
    try:
        import reportlab  # noqa: F401
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "reportlab"])


# PDF colour palette (green & gold)
PDF_TITLE = "#0b3d2a"
PDF_HEAD = "#0f5132"
PDF_GOLD = "#d4a017"
PDF_ZEBRA = "#eef6f0"
PDF_GRID = "#cfe3d6"
PDF_MUTED = "#5f7a6b"
PDF_TEXT = "#12281d"
PDF_FOOT = "#7a9585"


def _pdf_styles():
    from reportlab.lib.styles import ParagraphStyle
    from reportlab.lib import colors
    base = dict(fontName="Helvetica", textColor=colors.HexColor(PDF_TEXT))
    return {
        "title": ParagraphStyle("t", fontName="Helvetica-Bold", fontSize=20, leading=24, textColor=colors.HexColor(PDF_TITLE), alignment=1),
        "sub": ParagraphStyle("s", fontSize=9.5, leading=13, textColor=colors.HexColor(PDF_MUTED), fontName="Helvetica", alignment=1),
        "cell": ParagraphStyle("c", fontSize=8.5, leading=11, **base),
        "cellc": ParagraphStyle("cc", fontSize=8.5, leading=11, alignment=1, **base),
        "head": ParagraphStyle("h", fontName="Helvetica-Bold", fontSize=8.5, leading=11, textColor=colors.white),
        "headc": ParagraphStyle("hc", fontName="Helvetica-Bold", fontSize=8.5, leading=11, textColor=colors.white, alignment=1),
        "note": ParagraphStyle("n", fontSize=8, leading=11, textColor=colors.HexColor(PDF_MUTED), fontName="Helvetica", alignment=1),
    }


def _pdf_table_style(n_rows, header_color=PDF_HEAD):
    from reportlab.platypus import TableStyle
    from reportlab.lib import colors
    cmds = [("BACKGROUND", (0, 0), (-1, 0), colors.HexColor(header_color)),
            ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
            ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor(PDF_GRID)),
            ("LINEBELOW", (0, 0), (-1, 0), 1.2, colors.HexColor(PDF_GOLD)),
            ("TOPPADDING", (0, 0), (-1, -1), 5), ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
            ("LEFTPADDING", (0, 0), (-1, -1), 6), ("RIGHTPADDING", (0, 0), (-1, -1), 6)]
    for i in range(2, n_rows, 2):
        cmds.append(("BACKGROUND", (0, i), (-1, i), colors.HexColor(PDF_ZEBRA)))
    return TableStyle(cmds)


def _footer(canvas, doc):
    from reportlab.lib import colors
    canvas.saveState()
    canvas.setFont("Helvetica", 8)
    canvas.setFillColor(colors.HexColor(PDF_FOOT))
    w = doc.pagesize[0]
    canvas.drawString(doc.leftMargin, 22, "Generated " + datetime.now().strftime("%B %d, %Y %I:%M %p"))
    canvas.drawRightString(w - doc.rightMargin, 22, f"Page {doc.page}")
    canvas.restoreState()


def make_report_pdf(path, s):
    ensure_reportlab()
    from reportlab.lib.pagesizes import A4
    from reportlab.lib import colors
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, HRFlowable
    st = _pdf_styles()
    doc = SimpleDocTemplate(path, pagesize=A4, leftMargin=36, rightMargin=36, topMargin=40, bottomMargin=44,
                            title=f"Grade Report - {student_info['name'] or ''}")
    width = A4[0] - 72
    story = [Paragraph("Grade Report", st["title"]),
             Paragraph(esc(current_file["term"]) or "&nbsp;", st["sub"]),
             Spacer(1, 4), HRFlowable(width="100%", thickness=1.2, color=colors.HexColor(PDF_GOLD)), Spacer(1, 8)]

    meta = [("Name", student_info["name"]), ("Student No.", student_info["student_no"]),
            ("Program", student_info["course"]), ("Year level", student_info["year_level"])]
    meta = [(k, v) for k, v in meta if v]
    mrows = [[Paragraph(f"<b>{k}</b>", st["cell"]), Paragraph(esc(v), st["cell"])] for k, v in meta]
    mt = Table(mrows, colWidths=[80, width - 80])
    mt.setStyle([("TOPPADDING", (0, 0), (-1, -1), 2), ("BOTTOMPADDING", (0, 0), (-1, -1), 2), ("LEFTPADDING", (0, 0), (-1, -1), 0)])
    story += [mt, Spacer(1, 12)]

    heads, widths = [], []
    if s["has_code"]:
        heads.append(("Code", "head")); widths.append(58)
    heads.append(("Subject", "head")); widths.append(None)
    if s["has_teacher"]:
        heads.append(("Teacher", "head")); widths.append(105)
    heads += [("Units", "headc"), ("Grade", "headc"), ("Evaluation", "head"), ("Remark", "headc")]
    widths += [34, 38, 64, 54]
    fixed = sum(w for w in widths if w)
    widths = [w if w else width - fixed for w in widths]
    data = [[Paragraph(h, st[k]) for h, k in heads]]
    for r in s["rows"]:
        remark = esc(r["remark"])
        rl = r["remark"].lower()
        col = "#15803d" if rl == "passed" else ("#b91c1c" if rl == "failed" else "#a16207")
        row = []
        if s["has_code"]:
            row.append(Paragraph(esc(r["code"]), st["cell"]))
        row.append(Paragraph(esc(r["subject"]), st["cell"]))
        if s["has_teacher"]:
            row.append(Paragraph(esc(r["teacher"]), st["cell"]))
        row += [Paragraph(f"{r['units']:g}", st["cellc"]), Paragraph(esc(fmt_grade(r["grade"])), st["cellc"]),
                Paragraph(esc(r.get("quality", grade_quality(r["grade"]))), st["cell"]),
                Paragraph(f'<font color="{col}"><b>{remark}</b></font>', st["cellc"])]
        data.append(row)
    t = Table(data, colWidths=widths, repeatRows=1)
    t.setStyle(_pdf_table_style(len(data)))
    story += [t, Spacer(1, 16)]

    kp = [("GWA · " + grade_quality(s["gwa"]), fmt_num(s["gwa"])), ("Total units", f"{s['total_units']:g}"), ("Subjects", str(s["subjects"])),
          ("Best", fmt_num(s["best"])), ("Lowest", fmt_num(s["worst"])), ("Failed", str(s["failed"])),
          ("Dean's Lister", "Yes" if s["dean_list"] else "No")]
    lab = st["note"]
    kt = Table([[Paragraph(k, lab) for k, _ in kp],
                [Paragraph(f"<b><font size=14>{v}</font></b>", st["cell"]) for _, v in kp]],
               colWidths=[width / len(kp)] * len(kp))
    kt.setStyle([("BACKGROUND", (0, 0), (-1, -1), colors.HexColor(PDF_ZEBRA)), ("BOX", (0, 0), (-1, -1), 0.6, colors.HexColor(PDF_GRID)),
                 ("LINEAFTER", (0, 0), (-2, -1), 0.4, colors.HexColor(PDF_GRID)),
                 ("ALIGN", (0, 0), (-1, -1), "CENTER"), ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
                 ("LEFTPADDING", (0, 0), (-1, -1), 4), ("RIGHTPADDING", (0, 0), (-1, -1), 4),
                 ("TOPPADDING", (0, 0), (-1, -1), 8), ("BOTTOMPADDING", (0, 0), (-1, -1), 8)])
    story += [kt, Spacer(1, 10),
              Paragraph(f"Grading scale: 1.00 = highest, {PASSING_GRADE:.2f} = lowest passing, 5.00 = failed. "
                        "GWA = sum(grade x units) / total units. "
                        f"Dean's Lister: GWA of {DEANS_LIST_THRESHOLD:.2f} or better with no grade worse than {DEANS_LIST_MAX_GRADE:.2f}.", st["note"])]
    doc.build(story, onFirstPage=_footer, onLaterPages=_footer)


def make_table_pdf(path, title, subtitle, headers, rows, note="", page=None, font=8):
    """Generic landscape table PDF used by the class summary and the filtered-data export."""
    ensure_reportlab()
    from reportlab.lib.pagesizes import A4, A3, landscape
    from reportlab.lib.styles import ParagraphStyle
    from reportlab.lib import colors
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, HRFlowable
    st = _pdf_styles()
    size = page or landscape(A4)
    doc = SimpleDocTemplate(path, pagesize=size, leftMargin=30, rightMargin=30, topMargin=36, bottomMargin=42, title=title)
    width = size[0] - 60
    cell = ParagraphStyle("tc", parent=st["cell"], fontSize=font, leading=font + 2.5)
    head = ParagraphStyle("th", parent=st["head"], fontSize=font, leading=font + 2.5)
    data = [[Paragraph(esc(h), head) for h in headers]] + [[Paragraph(esc(v), cell) for v in r] for r in rows]
    t = Table(data, colWidths=[width / len(headers)] * len(headers), repeatRows=1)
    t.setStyle(_pdf_table_style(len(data)))
    story = [Paragraph(esc(title), st["title"]), Paragraph(esc(subtitle) or "&nbsp;", st["sub"]), Spacer(1, 4),
             HRFlowable(width="100%", thickness=1.2, color=colors.HexColor(PDF_GOLD)), Spacer(1, 10), t]
    if note:
        story += [Spacer(1, 8), Paragraph(esc(note), st["note"])]
    doc.build(story, onFirstPage=_footer, onLaterPages=_footer)


def do_download_report(e):
    state, payload = current_summary()
    if state != "ok" or not payload["rows"]:
        report_status.value = '<div class="status-msg status-error" style="margin:0;">Select a student with grade records first.</div>'
        return
    out_name = f"grade_report_{safe_filename(student_info['name'], 'student')}.pdf"
    try:
        make_report_pdf(out_name, payload)
    except Exception as exc:
        report_status.value = f'<div class="status-msg status-error" style="margin:0;">Could not build the PDF: {esc(exc)}</div>'
        return
    deliver_file(out_name, report_status)


def do_download_class(e):
    if current_file["cols"] is None or not current_file["students"]:
        class_status.value = '<div class="status-msg status-error" style="margin:0;">No class data to export.</div>'
        return
    view = class_view(dd_sort.value, dd_initial.value, dd_eval.value)
    if not view:
        class_status.value = '<div class="status-msg status-error" style="margin:0;">No students match the current filters.</div>'
        return
    heads = ["Student No.", "Student", "Program", "Subjects", "Units", "GWA", "Grade Quality", "Best", "Lowest", "Failed", "Evaluation"]
    rows = [[c["student_no"], c["name"], c["program"], c["subjects"], f"{c['total_units']:g}", fmt_num(c["gwa"]),
             grade_quality(c["gwa"]), fmt_num(c["best"]), fmt_num(c["worst"]), c["failed"], evaluation(c)] for c in view]
    labels = dict((v, k) for k, v in SORT_OPTIONS), dict((v, k) for k, v in EVAL_OPTIONS)
    applied = [f"Sorted: {labels[0][dd_sort.value]}"]
    if dd_initial.value != "All":
        applied.append(f"Starts with {dd_initial.value}")
    if dd_eval.value != "all":
        applied.append(labels[1][dd_eval.value])
    subtitle = " · ".join(x for x in (current_file["name"], current_file["term"], f"{len(view)} of {len(current_file['students'])} students") if x)
    base = safe_filename((current_file["name"] or "grades").rsplit(".", 1)[0], "grades")
    out_name = f"class_summary_{base}.pdf"
    try:
        make_table_pdf(out_name, "Class Summary", subtitle, heads, rows,
                       note=" · ".join(applied) + f". Dean's Lister: GWA of {DEANS_LIST_THRESHOLD:.2f} or better with no grade worse than {DEANS_LIST_MAX_GRADE:.2f}.")
    except Exception as exc:
        class_status.value = f'<div class="status-msg status-error" style="margin:0;">Could not build the PDF: {esc(exc)}</div>'
        return
    deliver_file(out_name, class_status)


def do_download(e):
    from reportlab.lib.pagesizes import A3, landscape
    df = filtered_df()
    if df is None or df.empty:
        status.value = '<div class="status-msg status-error">Nothing to download yet.</div>'
        return
    cap = 1500
    shown = df.head(cap).fillna("")
    base = safe_filename((current_file["name"] or "data").rsplit(".", 1)[0], "data")
    out_name = f"filtered_{base}.pdf"
    sub = f"{len(df):,} rows" + (f" matching '{table_state['search']}'" if table_state["search"] else "")
    note = f"Showing the first {cap:,} of {len(df):,} rows." if len(df) > cap else ""
    many = len(df.columns) > 10
    try:
        make_table_pdf(out_name, "Data Export", sub, [str(c) for c in shown.columns],
                       [[("" if v == "" else str(v)) for v in r] for r in shown.itertuples(index=False)], note=note,
                       page=landscape(A3) if many else None, font=7 if many else 8)
    except Exception as exc:
        status.value = f'<div class="status-msg status-error">Could not build the PDF: {esc(exc)}</div>'
        return
    deliver_file(out_name, status)


btn_download_report.on_click(do_download_report)
btn_download_class.on_click(do_download_class)
btn_download.on_click(do_download)


# ============================================================
# START
# ============================================================
# Preview card is only useful once a file is loaded; show it as a tab, with its empty state.
render_all()
display(app_container)